# Importing required libraries

In [1]:
import os
import csv
import requests
import json
import base64
from difflib import SequenceMatcher
import pandas as pd
import matplotlib.pyplot as plt

# Configurating LM-Studio

In [2]:
LMSTUDIO_URL = "http://127.0.0.1:1234/v1/chat/completions"
API_KEY = "sk-lm-c3muNwmH:WrSpN530w8xOHMF1rncq"
MODEL = "llava-phi-3-mini"
DATASET_PATH = r"C:\Users\USER\Documents\LicensePlateOCR\Indonesian License Plate Dataset\images\test"
OUTPUT_CSV = "results.csv"

# CER FUNCTION

In [3]:
def cer_score(prediction: str, ground_truth: str) -> float:
    matcher = SequenceMatcher(None, prediction, ground_truth)
    S = D = I = 0
    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        if tag == 'replace':
            S += max(i2 - i1, j2 - j1)
        elif tag == 'delete':
            D += (i2 - i1)
        elif tag == 'insert':
            I += (j2 - j1)
    N = len(ground_truth)
    return (S + D + I) / N if N > 0 else 0.0

# LM-Studio

In [4]:
def query_lmstudio(image_path: str) -> str:
    with open(image_path, "rb") as f:
        img_b64 = base64.b64encode(f.read()).decode("utf-8")

    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": MODEL,
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": "What is the license plate number shown in this image? Respond only with the plate number."},
                    {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{img_b64}"}}
                ]
            }
        ]
    }

    resp = requests.post(LMSTUDIO_URL, headers=headers, data=json.dumps(payload))
    resp.raise_for_status()
    result = resp.json()
    return result["choices"][0]["message"]["content"].strip()

In [5]:
results = []
for img_file in os.listdir(DATASET_PATH):
    if not img_file.lower().endswith(('.png', '.jpg', '.jpeg')):
        continue

    img_path = os.path.join(DATASET_PATH, img_file)
    ground_truth = os.path.splitext(img_file)[0]
    try:
        prediction = query_lmstudio(img_path)
        cer = cer_score(prediction, ground_truth)
        results.append((img_file, prediction, ground_truth, cer))
    except Exception as e:
        print(f"Error processing {img_file}: {e}")
        results.append((img_file, "Error", ground_truth, None))

# save results

In [6]:
with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    # tulis header
    writer.writerow(["image", "ground_truth", "prediction", "CER_score"])
    # tulis isi hasil
    for row in results:
        writer.writerow(row)

print(f"Hasil prediksi disimpan ke {OUTPUT_CSV}")

Hasil prediksi disimpan ke results.csv


# load results and visualize

In [7]:
df = pd.DataFrame(results, columns=["image", "ground_truth", "prediction", "CER_score"])
df.head()

,image,ground_truth,prediction,CER_score
0,test001.jpg,"The image captures a moment on a rainy day, wh...",test001,180.142857
1,test002.jpg,The image depicts a scene on a highway. The ma...,test002,107.428571
2,test003.jpg,"The image captures a bustling city street, tee...",test003,144.000000
3,test004.jpg,The image depicts a scene on a highway where a...,test004,58.857143
4,test005.jpg,The image depicts a scene on a road. The main ...,test005,109.428571


In [ ]:
plt.figure(figsize=(8,5))
plt.hist(df["CER_score"], bins=20, color="skyblue", edgecolor="black")
plt.title("CER Distribution across Test Images")
plt.xlabel("CER")
plt.ylabel("Frequency")
plt.show()

# per image CER bar chart

In [ ]:
plt.figure(figsize=(20,10))  # larger canvas for breathing room
plt.bar(df["image"], df["CER_score"], color="salmon", edgecolor="black", alpha=0.8)

# Rotate labels and spread them out
plt.xticks(rotation=75, ha="right", fontsize=12)
plt.yticks(fontsize=12)

# Titles and labels with extra spacing
plt.title("CER per Image", fontsize=18, pad=25)
plt.xlabel("Image", fontsize=14, labelpad=15)
plt.ylabel("CER", fontsize=14, labelpad=15)

# Add grid lines for clarity
plt.grid(axis="y", linestyle="--", alpha=0.6)

plt.tight_layout()
plt.show()



# Results.csv

In [ ]:
df = pd.read_csv("results.csv")
print(df.head())
print(df.describe())